# Tutorial 3, Q1: LP, ILP and MILP

maximise 12x + 18y subject to 2x + 4y ≤ 55, 2x + y ≤ 25, x, y ≥ 0

- **Q1**: solve the LP with Pyomo
- **Q1a**: x and y integer (ILP)
- **Q1b**: x real, y integer (MILP)

Release after the practical. One model is built once and re-solved with different variable domains.

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes about a minute), then *Run → Run All Cells*, or run the cells one by one while you read.

**Expected result.** LP: 270 at (7.5, 10). ILP: 264 at (7, 10). MILP: 270 at (7.5, 10).

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

`solve()` builds the model for given variable domains (real or integer), solves it with HiGHS and prints the answer. It is called three times: LP, ILP and MILP.

In [ ]:
import pyomo.environ as pyo

def solve(x_domain, y_domain, label):
    m = pyo.ConcreteModel()
    m.x = pyo.Var(domain=x_domain)      # real or integer, as requested
    m.y = pyo.Var(domain=y_domain)
    m.obj = pyo.Objective(expr=12 * m.x + 18 * m.y, sense=pyo.maximize)
    m.c1 = pyo.Constraint(expr=2 * m.x + 4 * m.y <= 55)  # 2x + 4y <= 55
    m.c2 = pyo.Constraint(expr=2 * m.x + m.y <= 25)      # 2x + y <= 25
    result = pyo.SolverFactory("appsi_highs").solve(m)
    print(f"{label:<22} x = {pyo.value(m.x):6.3f}   y = {pyo.value(m.y):6.3f}   "
          f"objective = {pyo.value(m.obj):7.2f}   ({result.solver.termination_condition})")
    return pyo.value(m.x), pyo.value(m.y)

lp   = solve(pyo.NonNegativeReals,    pyo.NonNegativeReals,    "Q1  LP")
ilp  = solve(pyo.NonNegativeIntegers, pyo.NonNegativeIntegers, "Q1a ILP (x, y integer)")
milp = solve(pyo.NonNegativeReals,    pyo.NonNegativeIntegers, "Q1b MILP (y integer)")

## Check graphically

The shaded area is the LP's feasible region; dots are the integer points inside it. The dashed line through
the LP optimum is the objective contour 12x + 18y = constant.

Plot the feasible region, the integer points and the three optima, as the sheet suggests.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

xs = np.linspace(0, 14, 400)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(xs, (55 - 2 * xs) / 4, label="2x + 4y = 55")
ax.plot(xs, 25 - 2 * xs, label="2x + y = 25")
# shade the LP feasible region: below both lines and above y = 0
ax.fill_between(xs, 0, np.minimum((55 - 2 * xs) / 4, 25 - 2 * xs).clip(min=0), alpha=0.15)

# all whole-number points (x, y) that satisfy both constraints
pts = [(i, j) for i in range(14) for j in range(15) if 2 * i + 4 * j <= 55 and 2 * i + j <= 25]
ax.scatter(*zip(*pts), s=8, color="grey", label="integer points")
ax.plot(xs, (12 * lp[0] + 18 * lp[1] - 12 * xs) / 18, "k--", lw=0.8, label="objective at LP optimum")
ax.scatter(*lp, s=80, marker="o", label="LP optimum", zorder=3)
ax.scatter(*ilp, s=80, marker="s", label="ILP optimum", zorder=3)
ax.scatter(*milp, s=80, marker="^", label="MILP optimum", zorder=3)
ax.set(xlim=(0, 14), ylim=(0, 15), xlabel="x", ylabel="y", title="Tutorial 3, Q1")
ax.legend(fontsize=8)
plt.show()

## What do you notice?

In the plot you can see the square (ILP) and the triangle (MILP), but not the circle (LP). Why? And why does
the ILP lose 6 while the MILP loses nothing?

<details><summary>Answer</summary>

The LP and MILP optima are the same point, (7.5, 10), so the triangle hides the circle. The LP optimum already
has a whole-number $y = 10$, so requiring $y$ to be an integer (MILP) changes nothing. Requiring $x$ to be an
integer as well (ILP) moves the answer to (7, 10), worth $12 \times 0.5 = 6$ less.
</details>